In [ ]:
!curl -o kural.txt https://raw.githubusercontent.com/b1zantine/thirukkural-dataset/master/thirukkural.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  218k  100  218k    0     0   924k      0 --:--:-- --:--:-- --:--:--  925k


In [ ]:
with open("kural.txt","r") as f:
  data = f.read()

In [ ]:
from collections import defaultdict

In [ ]:
data[:100]

'அகர முதல எழுத்தெல்லாம் ஆதி$பகவன் முதற்றே உலகு.\nகற்றதனால் ஆய பயனென்கொல் வாலறிவன்$நற்றாள் தொழாஅர் எனின'

In [ ]:
bt = data.encode('utf-8')
toks = list(map(int, bt))

In [ ]:
len(toks)

223652

In [ ]:
def count_pairs(ids):
  counts = defaultdict(int)

  for pair in zip(ids, ids[1:]):
    counts[pair] += 1

  return counts

def replace(ids, pair, new_tok):
  newli = []
  i = 0
  while i < len(ids):
    if i < len(ids)-1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newli.append(new_tok)
      i += 2
    else:
      newli.append(ids[i])
      i+=1

  return newli

In [ ]:
# train loop
new_tok = 5000
rdata = toks
tokenizer_mer = {}

In [ ]:
for i in range(new_tok):
  stat = count_pairs(rdata)
  pair = max(stat, key = lambda p: stat[p])
  tok = 256 + i
  re = replace(rdata, pair, tok)
  rdata = re
  tokenizer_mer[pair] = tok

In [ ]:
#tokenizer_mer

In [ ]:
vocab = {i: bytes([i]) for i in range(256)}
for (p0, p1), tokens in tokenizer_mer.items():
  vocab[tokens] = vocab[p0] + vocab[p1]

In [ ]:
#vocab

In [ ]:
def encode(text):
  token = list(text.encode('utf-8'))
  while len(text) > 2:
    get_stat = count_pairs(token)
    pair = min(get_stat, key= lambda p:tokenizer_mer[p] if p in tokenizer_mer else float('inf'))
    if pair not in tokenizer_mer:
      break
    tok = tokenizer_mer[pair]
    token = replace(token, pair, tok)

  return token

In [ ]:
def decode(li):
  raw = b''.join(vocab[i] for i in li)
  return raw.decode('utf-8')

In [ ]:
len(vocab)

10256

In [ ]:
len(encode("தெய்வத்தான் ஆகா தெனினும் முயற்சிதன் மெய்வருத்தக் கூலி தரும்")) # kutti-bpe - my tokenizer

16

In [ ]:
decode(encode("தெய்வத்தான் ஆகா தெனினும் முயற்சிதன் மெய்வருத்தக் கூலி தரும்"))

'தெய்வத்தான் ஆகா தெனினும் முயற்சிதன் மெய்வருத்தக் கூலி தரும்'

In [ ]:
stream = "அண்மையில் நடைபெற்ற சர்வதேச அறிவியல் தொழில்நுட்ப மாநாட்டில் உலகெங்கிலுமிருந்து வருகை தந்திருந்த முன்னணி பேராசிரியர்களும் ஆராய்ச்சியாளர்களும் தங்களது நவீன கண்டுபிடிப்புகளையும் ஆய்வுக்கட்டுரைகளையும் சமர்ப்பித்ததோடு மட்டுமன்றி மனிதகுல முன்னேற்றத்திற்கான எதிர்கால திட்டங்கள் குறித்தும் மிக விரிவாக விவாதித்தனர் குறிப்பாக செயற்கை நுண்ணறிவு மற்றும் இயந்திரவழிக் கற்றல் போன்ற மேம்பட்ட தொழில்நுட்பங்கள் தற்கால மருத்துவத்துறையிலும் கல்வித்துறையிலும் எத்தகைய வியக்கத்தக்க புரட்சிகரமான மாற்றங்களை ஏற்படுத்தி வருகின்றன என்பதைப் பலதரப்பட்ட சான்றுகளுடன் விளக்கிய அவர்கள் இத்தகைய நவீன தொழில்நுட்பங்களை கிராமப்புற மாணவர்களும் எளிய மக்களும் எளிதாகப் பயன்படுத்திக்கொள்ளும் வகையில் அரசு உரிய நடவடிக்கைகளை உடனடியாக எடுக்க வேண்டுமென வலியுறுத்தினர் மேலும் சுற்றுச்சூழல் பாதுகாப்பு மற்றும் புதுப்பிக்கத்தக்க ஆற்றல் வளங்களின் முக்கியத்துவத்தை உணர்ந்து அனைவரும் ஒன்றிணைந்து செயல்பட வேண்டியதன் அவசியத்தையும் இந்த மாநாடு உலகிற்கு மிகத் தெளிவாக உணர்த்தியது."
splitted = stream.split()

In [ ]:
import numpy as np

In [ ]:
np.mean([len(encode(i)) for i in splitted])

np.float64(4.988372093023256)

In [ ]:
fertility = sum([len(encode(i)) for i in splitted]) / len(splitted)
fertility

4.988372093023256

In [ ]:
encode("பயன்படுத்திக்கொள்ளும்"), len(encode("பயன்படுத்திக்கொள்ளும்"))

([313, 280, 362, 1195, 542, 864, 276, 258], 8)

In [ ]:
len(encode("தமிழை உலகமெங்கும் கொண்டு சேர்ப்போம்."))

15